# 快速体验 LangChain

主要内容：
用阿里百炼的 OpenAI 兼容接口，快速跑通一个最小的 LangChain Agent 闭环。

整体流程很简单：
1. 先直接调用一次大模型，感受它本身的能力边界。
2. 再定义一个工具，让模型可以借助外部能力获取信息。
3. 最后交给 Agent 统一编排，由 Agent 决定什么时候调用工具、什么时候直接回复。

这个例子不追求完整覆盖 LangChain 的全部能力，只关注一个最小切入点：
- 模型先独立回答一次
- 再接入一个工具
- 最后交给 Agent 编排

这样更容易看清：模型、工具、Agent 三者各自负责什么。


In [ ]:
# 使用模型云服务商，以阿里百炼为例

import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
completion = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[{'role': 'user', 'content': '上海今天天气如何'}]
)
print(completion.choices[0].message.content)

上面这一步的目的，是先观察“只有模型、不接工具”时的回答边界。

对于天气这类实时信息，模型本身通常无法直接给出可靠结果，因此下一步需要引入工具。


大模型本身无法主动调用外部工具。如果想获取实时信息，比如天气，就可以让 Agent 配合工具来完成。

一、安装依赖

```bash
uv add langchain langchain-openai python-dotenv
```

这里选择 `langchain-openai`，因为阿里百炼提供的是 OpenAI 兼容接口。

In [ ]:
# 二、加载环境变量
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# 三、定义一个最简单的工具
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """获取指定城市的天气信息。"""
    return f"{location}今天天气晴朗，气温 22°C。"

有了模型和工具之后，还需要一个调度层来决定：
- 什么时候直接回答
- 什么时候调用工具

这就是 Agent 在这个例子里的作用。


In [ ]:
# 四、定义Agent
import os
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="qwen3.5-122b-a10b",
    api_key=os.environ.get('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

agent = create_agent(
    model=model,
    tools=[get_weather]
)

In [ ]:
# 五、调用Agent
response = agent.invoke({
    "messages":[
        {"role":"user","content":"上海今天天气怎么样？"}
    ]
})
print(response["messages"][-1].content)